# Building chatbot interfaces using Gradio, connecting to the Gemini API via OpenAI, and handling multi-turn conversation history.

In [ ]:
import gradio as gr

In [ ]:
def greet(message):
  return 'hello'

In [ ]:
from typing import Never
gr.Interface(fn=greet,inputs="textbox", outputs="textbox", flagging_mode='never').launch()

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
from google.colab import userdata

In [ ]:
api_key=userdata.get('GEMINI_API')

In [ ]:
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta",
    api_key=api_key
)

In [ ]:
system_prompt= 'You are a helpful assistant'

In [ ]:
def call_gemini(prompt, history):
    messages = [{"role": "system", "content": system_prompt}]

    # Format the history safely regardless of Gradio's internal format
    for msg in history:
        if isinstance(msg, dict):
            messages.append({"role": msg.get("role"), "content": msg.get("content")})
        elif isinstance(msg, (list, tuple)) and len(msg) >= 2:
            messages.append({"role": "user", "content": msg[0]})
            messages.append({"role": "assistant", "content": msg[1]})

    # Append the current user prompt
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages
    )
    return response.choices[0].message.content

In [ ]:
gr.Interface(fn=call_gemini).launch(debug=True)

In [ ]:
def call_gpt(prompt, message):
  return "GPT is called"

In [ ]:
model_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Gemini"], label="Select model", value= "GPT")
message_output= gr.Textbox(label= "Output", lines= 7)

In [ ]:
gr.Interface(fn=call_gpt, inputs= [model_input, model_selector], outputs= [message_output]).launch(debug=True, auth=("Gurdeep","005"))

In [ ]:
gr.ChatInterface(fn=call_gemini, type="messages").launch(debug=True)